# 쇼핑몰 상품 추천 RAG v3 (현재 상품 중심 + Profile 순위 보정)

v2는 거래 Profile 신호가 현재 상품 검색보다 더 큰 비중(질문 18% vs 거래신호 54%)을 차지해서,
"과거에 많이 팔린 상품"이 "지금 질문과 맞는 상품"을 밀어낼 수 있었습니다.

v3는 이 역할을 뒤집습니다: **현재 상품 검색을 추천의 중심에 두고, 거래 Profile은 순위를 보정하는
신호로만 사용**합니다. README.md "13. 다음 단계: v3 계획"에서 정한 7개 과제를 그대로 구현합니다.

## v2 대비 무엇이 바뀌는가

| 구분 | v2 | v3 |
|---|---|---|
| Profile 그룹 키 | 구매처+**상품분류**+주문월+행사/대상/시즌 | 구매처+주문월+행사/대상/시즌 (**상품분류 제거**) |
| Profile 내부 정보 | 대표상품(첫 등장 순) | 대표상품(**빈도 순**) + **상품분류별 거래건수/비율** |
| 가격/MOQ/판매상태 | soft score (위반해도 감점만) | **하드 필터** (위반 시 후보에서 제외, 완화 단계 포함) |
| 점수 비중 | 질문 18% vs 거래신호 54% | **질문 50%** vs Profile 보정 20% (역할 역전) |
| 관련도 임계값 | 없음 (top_k로만 자름) | **있음** (backend별 점수 스케일이 달라 상대(비율) 기준 적용) |
| 최근 거래 가중치 | 없음 | **있음** (최신 거래일 기준 지수 감쇠) |
| 후보 다양성 | 없음 | **있음** (같은 소분류 도배 방지) |
| 추천 후보 ID 검증 | 없음 | **있음** (LLM 답변에 후보 외 상품이 섞이는지 확인) |
| 정량 평가 | 없음 (정성 테스트만) | **있음** (Hit@5, 카테고리 키워드 기반 자동 정답셋 초안) |

## 이 샘플 데이터(100건)에서 미리 알아야 할 제약

- **거래 100건으로는 Profile 집계 효과가 크지 않습니다.** 구매처 세분류만 76종/100건일 정도로
  다양해서, 상품분류를 그룹 키에서 빼도 실제 감소율은 한 자릿수%대입니다. 이는 구현 오류가
  아니라 표본이 작아서 생기는 한계이며, 실제 운영 데이터(수만 건)에서는 반복 거래가 많아
  감소율이 커질 것으로 예상합니다.
- **거래데이터와 상품데이터의 상품분류 체계(taxonomy)가 서로 다릅니다.** 거래데이터는
  "생활용품", "문구/사무용품"처럼 큰 단위를, 상품데이터는 "볼펜/필기류", "USB메모리"처럼
  세부 단위를 씁니다. 그래서 카테고리 신호를 정확 일치로 비교하면 거의 매칭되지 않고,
  이 노트북에서는 상품 텍스트(카테고리경로+상품명+키워드) 안에 선호 카테고리 문자열이
  포함되는지로 겹침을 판단합니다.
- **판매상태(재고) 정보가 90%는 비어 있습니다.** 값이 없으면 "판매중으로 간주"하는
  보수적인 규칙을 씁니다. 품절이 확정된 경우(재고 0)만 실제로 제외됩니다.

## 이 노트북에서 보는 RAG 4단계

| 단계 | v2에서 하던 일 | v3에서 하는 일 |
|---|---|---|
| **1. Indexing** | 상품 row + 거래 Profile 임베딩 | 동일 + Profile에 상품분류 비율/최근성 필드 추가 |
| **2. Query 이해** | LLM 조건 추출 | 동일 |
| **3. Retrieval** | 유사 Profile 검색 → 신호텍스트 임베딩 비교 | 현재 상품 **하드 필터** → 유사 Profile 검색 → **카테고리 비율 신호** 추출 |
| **4. Ranking/Generation** | 거래신호 중심 재정렬 → LLM 설명 | **질문 중심** 재정렬 + 관련도 임계값 + 다양성 조정 → LLM 설명 → **후보 ID 검증** |

## 0. 설치

v1/v2와 동일합니다. 이미 실행해보셨다면 건너뛰어도 됩니다.

```bash
python -m pip install pandas openpyxl scikit-learn numpy ollama
ollama pull bge-m3
ollama pull gemma3:4b
```

In [ ]:
# 필요 시 한 번만 실행
# !pip install pandas openpyxl scikit-learn numpy ollama

## 1. 기본 설정 및 파일 경로

In [ ]:
from pathlib import Path
import re
import html
import json
import hashlib
import warnings

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 180)

PRODUCT_PATH = Path("../data/상품데이터_샘플100.xlsx")
TRADE_PATH = Path("../data/거래데이터_샘플100.xlsx")

OUTPUT_DIR = Path("../output")
CACHE_DIR = Path("../cache")
OUTPUT_DIR.mkdir(exist_ok=True)
CACHE_DIR.mkdir(exist_ok=True)

USE_OLLAMA_EMBEDDING = True
EMBED_MODEL = "bge-m3"
LLM_MODEL = "gemma3:4b"

DEFAULT_TOP_K = 5

print("PRODUCT_PATH:", PRODUCT_PATH)
print("TRADE_PATH:", TRADE_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())

## 2. 파일 로딩 및 컬럼 확인

In [ ]:
product_raw = pd.read_excel(PRODUCT_PATH, dtype=str).fillna("")
trade_raw = pd.read_excel(TRADE_PATH, dtype=str).fillna("")

print("상품데이터 shape:", product_raw.shape)
print("거래데이터 shape:", trade_raw.shape)
display(product_raw.head(3))
display(trade_raw.head(3))

## 3. 공통 전처리 함수

*`top_values_by_frequency`가 v3에서 새로 추가된 함수입니다. 기존 `join_top_values`는
"처음 등장한 순서"로 값을 뽑아서, 대표상품이 실제 거래 빈도와 다를 수 있다는 문제가
README 6.3에 지적되어 있었습니다.*

In [ ]:
def clean_text(x):
    """HTML 태그/엔티티를 제거하고 공백을 정리합니다."""
    if pd.isna(x):
        return ""
    x = str(x)
    x = re.sub(r"<[^>]+>", " ", x)
    x = html.unescape(x)
    x = re.sub(r"\s+", " ", x).strip()
    return x


def to_number(x):
    """'3,000원' 같은 문자열에서 숫자만 뽑아 float로 변환합니다."""
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    if s in ["", "-", "nan", "None", "null"]:
        return np.nan
    s = re.sub(r"[^0-9.]", "", s)
    if s == "":
        return np.nan
    try:
        return float(s)
    except Exception:
        return np.nan


def safe_col(df, col):
    """컬럼이 없으면 빈 문자열 시리즈를 돌려줘서 KeyError를 막습니다."""
    if col in df.columns:
        return df[col].fillna("").astype(str)
    return pd.Series([""] * len(df), index=df.index)


def combine_text(*parts):
    return clean_text(" ".join([str(p) for p in parts if str(p).strip()]))


def join_top_values(series, n=8):
    """값을 중복 제거 후 '처음 등장한 순서'로 상위 n개만 이어 붙입니다."""
    values = [clean_text(x) for x in series.tolist() if clean_text(x)]
    values = list(dict.fromkeys(values))
    return " / ".join(values[:n])


def top_values_by_frequency(series, n=8):
    """v3: 대표값을 '처음 등장 순'이 아니라 '거래 빈도 순'으로 뽑습니다."""
    values = [clean_text(x) for x in series.tolist() if clean_text(x)]
    if not values:
        return ""
    counts = pd.Series(values).value_counts()
    return " / ".join(counts.head(n).index.tolist())


def normalize_vector_matrix(mat: np.ndarray) -> np.ndarray:
    """임베딩 행렬을 행 단위 L2 정규화합니다 (코사인 유사도를 내적으로 계산하기 위함)."""
    mat = np.array(mat, dtype=np.float32)
    norms = np.linalg.norm(mat, axis=1, keepdims=True)
    norms[norms == 0] = 1
    return mat / norms


def cosine_scores(query_vec: np.ndarray, matrix: np.ndarray) -> np.ndarray:
    """정규화된 query 벡터와 문서 행렬 간 코사인 유사도(=내적)를 계산합니다."""
    query_vec = np.array(query_vec, dtype=np.float32)
    if query_vec.ndim == 1:
        query_vec = query_vec.reshape(1, -1)
    query_norm = np.linalg.norm(query_vec, axis=1, keepdims=True)
    query_norm[query_norm == 0] = 1
    query_vec = query_vec / query_norm
    return (matrix @ query_vec.T).ravel()


def text_hash(texts):
    """임베딩 캐시 파일명을 만들기 위한 내용 해시입니다."""
    joined = "\n".join([str(x) for x in texts])
    return hashlib.md5(joined.encode("utf-8")).hexdigest()[:10]

## 4. 상품데이터 정규화 (+ 판매상태)

*v3 신규: `재고` 컬럼을 기반으로 `is_sold_out`을 만듭니다. 이 샘플 데이터는 재고 값이
90%는 비어 있어서, "비어 있으면 판매중으로 간주"하는 보수적인 규칙을 씁니다 — 값이 명확히
0인 경우만 품절로 확정합니다.*

In [ ]:
product_df = product_raw.copy()

product_df["product_id"] = safe_col(product_df, "상품번호")
product_df.loc[product_df["product_id"].str.strip() == "", "product_id"] = safe_col(product_df, "상품코드")

product_df["brand"] = safe_col(product_df, "브랜드").map(clean_text)
product_df["product_name"] = safe_col(product_df, "상품명").map(clean_text)
product_df["model_name"] = safe_col(product_df, "모델명").map(clean_text)

product_df["price"] = safe_col(product_df, "상품판매가").map(to_number)
product_df["moq"] = safe_col(product_df, "최소구매수량").map(to_number)

# v3 신규: 판매상태 (하드 필터에 사용)
product_df["stock_raw"] = safe_col(product_df, "재고").map(clean_text)
product_df["stock_num"] = product_df["stock_raw"].map(to_number)
product_df["is_sold_out"] = product_df["stock_num"] == 0

product_df["category_large"] = safe_col(product_df, "대 카테고리").map(clean_text)
product_df["category_middle"] = safe_col(product_df, "중 카테고리").map(clean_text)
product_df["category_small"] = safe_col(product_df, "소 카테고리").map(clean_text)
product_df["category_detail"] = safe_col(product_df, "세분류").map(clean_text)

product_df["category_path"] = (
    product_df["category_large"] + " > " +
    product_df["category_middle"] + " > " +
    product_df["category_small"] + " > " +
    product_df["category_detail"]
).map(clean_text)

product_df["keywords"] = safe_col(product_df, "검색키워드").map(clean_text)
product_df["summary"] = safe_col(product_df, "간략한설명").map(clean_text)
product_df["detail_text"] = safe_col(product_df, "상품내용").map(clean_text)
product_df["manufacturer"] = safe_col(product_df, "제조사").map(clean_text)
product_df["origin"] = safe_col(product_df, "원산지").map(clean_text)
product_df["image"] = safe_col(product_df, "큰이미지").map(clean_text)

product_df["product_search_text"] = (
    "[상품명] " + product_df["product_name"] + "\n" +
    "[브랜드] " + product_df["brand"] + "\n" +
    "[모델명] " + product_df["model_name"] + "\n" +
    "[카테고리] " + product_df["category_path"] + "\n" +
    "[검색키워드] " + product_df["keywords"] + "\n" +
    "[설명] " + product_df["summary"] + " " + product_df["detail_text"] + "\n" +
    "[제조/원산지] " + product_df["manufacturer"] + " " + product_df["origin"]
).map(clean_text)

product_df = product_df[product_df["product_name"].str.len() > 0].reset_index(drop=True)

print("정규화 상품 수:", len(product_df))
print("재고 값이 있는 상품 수:", (product_df["stock_raw"] != "").sum(), "/", len(product_df))
print("품절로 확정된 상품 수:", int(product_df["is_sold_out"].sum()))
display(product_df[["product_id", "product_name", "price", "moq", "stock_raw", "is_sold_out", "category_path"]].head(10))

## 5. 거래데이터 정규화 (row 단위)

*이 단계는 v2와 동일합니다. Profile 집계는 이 row 데이터를 바탕으로 다음 섹션에서 다시 만듭니다.*

In [ ]:
trade_df = trade_raw.copy()

buyer_name_col = "구매처 명 "
if buyer_name_col not in trade_df.columns and "구매처 명" in trade_df.columns:
    buyer_name_col = "구매처 명"

trade_df["buyer_type_large"] = safe_col(trade_df, "구매처 분류(대)").map(clean_text)
trade_df["buyer_type_middle"] = safe_col(trade_df, "구매처 분류(중)").map(clean_text)
trade_df["buyer_type_small"] = safe_col(trade_df, "구매처 분류(소)").map(clean_text)
trade_df["buyer_type_detail"] = safe_col(trade_df, "구매처 분류(세)").map(clean_text)
trade_df["buyer_name"] = safe_col(trade_df, buyer_name_col).map(clean_text)

trade_df["buyer_type_path"] = (
    trade_df["buyer_type_large"] + " > " +
    trade_df["buyer_type_middle"] + " > " +
    trade_df["buyer_type_small"] + " > " +
    trade_df["buyer_type_detail"]
).map(clean_text)

trade_df["date_raw"] = safe_col(trade_df, "날짜").map(clean_text)
trade_df["order_date"] = pd.to_datetime(trade_df["date_raw"], errors="coerce")
trade_df["order_month"] = trade_df["order_date"].dt.month

trade_df["trade_product_name"] = safe_col(trade_df, "상품").map(clean_text)

trade_df["trade_category_large"] = safe_col(trade_df, "상품분류(대)").map(clean_text)
trade_df["trade_category_middle"] = safe_col(trade_df, "상품분류(중)").map(clean_text)
trade_df["trade_category_small"] = safe_col(trade_df, "상품분류(소)").map(clean_text)

trade_df["trade_category_path"] = (
    trade_df["trade_category_large"] + " > " +
    trade_df["trade_category_middle"] + " > " +
    trade_df["trade_category_small"]
).map(clean_text)

trade_df["bulk_price"] = safe_col(trade_df, "대량가격(원)").map(to_number)
trade_df["middle_price"] = safe_col(trade_df, "중간가격(원)").map(to_number)
trade_df["small_price"] = safe_col(trade_df, "소량가격(원)").map(to_number)
trade_df["trade_moq"] = safe_col(trade_df, "최소구매수량").map(to_number)

trade_df["event_filter"] = safe_col(trade_df, "행사별(필터)").map(clean_text)
trade_df["target_filter"] = safe_col(trade_df, "대상별(필터)").map(clean_text)
trade_df["season_filter"] = safe_col(trade_df, "시즌별(필터)").map(clean_text)
trade_df["print_method"] = safe_col(trade_df, "인쇄 방법").map(clean_text)

trade_df = trade_df[trade_df["trade_product_name"].str.len() > 0].reset_index(drop=True)

DATASET_MAX_DATE = trade_df["order_date"].max()

print("정규화 거래 수 (row 단위):", len(trade_df))
print("데이터셋 내 최신 거래일 (recency 기준점):", DATASET_MAX_DATE)
display(trade_df[[
    "date_raw", "order_month", "buyer_type_path", "buyer_name",
    "trade_product_name", "trade_category_path",
    "bulk_price", "middle_price", "small_price", "trade_moq",
    "event_filter", "target_filter", "season_filter",
]].head(10))

## 6. 거래 Profile 재설계 (v3 핵심 1)

*RAG 단계: Indexing — v2는 그룹 키에 `trade_category_*`(상품분류)를 포함해서, 같은
구매상황(예: 병원+개원)이라도 상품이 다르면 Profile이 쪼개졌습니다(README 6.3).
v3는 상품분류를 그룹 키에서 빼고, 대신 Profile 내부에 "상품분류별 거래건수/비율"을
별도로 저장합니다. 대표상품도 첫 등장 순이 아니라 빈도 순으로 바꿉니다.*

```text
v2 그룹 키 = 구매처(4단계) + 상품분류(3단계) + 주문월 + 행사/대상/시즌
v3 그룹 키 = 구매처(4단계) +          주문월 + 행사/대상/시즌   (상품분류 제거)
```

또한 최근 거래 가중치(`recency_score`)를 추가합니다 — Profile 안에서 가장 최근 거래일이
데이터셋 내 최신 거래일보다 얼마나 오래됐는지에 따라 지수적으로 감쇠시킵니다
(`RECENCY_HALF_LIFE_DAYS`일이 지날 때마다 점수가 절반).

In [ ]:
PROFILE_GROUP_COLS = [
    "buyer_type_large", "buyer_type_middle", "buyer_type_small", "buyer_type_detail",
    "order_month", "event_filter", "target_filter", "season_filter",
]
PROFILE_GROUP_COLS = [c for c in PROFILE_GROUP_COLS if c in trade_df.columns]

RECENCY_HALF_LIFE_DAYS = 365  # 이 기간(일)이 지날 때마다 recency 점수가 절반으로 감소


def build_category_distribution(sub_df: pd.DataFrame, col: str) -> list:
    """profile 안에서 상품분류별 거래건수/비율을 리스트[dict]로 만듭니다 (README 6.4 과제)."""
    values = sub_df[col].map(clean_text)
    values = values[values != ""]
    if len(values) == 0:
        return []
    vc = values.value_counts()
    total = int(vc.sum())
    return [
        {"category": cat, "count": int(cnt), "ratio": round(cnt / total, 4)}
        for cat, cnt in vc.items()
    ]


def make_trade_profiles_v3(trade_df: pd.DataFrame) -> pd.DataFrame:
    """거래 row를 구매처/주문월/행사·대상·시즌 기준으로 묶습니다 (상품분류는 그룹 키에서 제외)."""
    rows = []
    for group_key, sub_df in trade_df.groupby(PROFILE_GROUP_COLS, dropna=False):
        if not isinstance(group_key, tuple):
            group_key = (group_key,)
        row = dict(zip(PROFILE_GROUP_COLS, group_key))

        row["trade_count"] = len(sub_df)
        row["sample_products"] = top_values_by_frequency(sub_df["trade_product_name"], n=10)
        row["sample_buyer_names"] = top_values_by_frequency(sub_df["buyer_name"], n=8)
        row["sample_print_methods"] = top_values_by_frequency(sub_df["print_method"], n=5)
        row["median_bulk_price"] = sub_df["bulk_price"].median()
        row["median_middle_price"] = sub_df["middle_price"].median()
        row["median_small_price"] = sub_df["small_price"].median()
        row["median_moq"] = sub_df["trade_moq"].median()

        row["category_middle_dist"] = build_category_distribution(sub_df, "trade_category_middle")
        row["category_small_dist"] = build_category_distribution(sub_df, "trade_category_small")
        row["last_order_date"] = sub_df["order_date"].max()

        rows.append(row)

    profile_df = pd.DataFrame(rows)

    profile_df["buyer_type_path"] = (
        profile_df["buyer_type_large"].fillna("").astype(str) + " > " +
        profile_df["buyer_type_middle"].fillna("").astype(str) + " > " +
        profile_df["buyer_type_small"].fillna("").astype(str) + " > " +
        profile_df["buyer_type_detail"].fillna("").astype(str)
    ).map(clean_text)

    profile_df["trade_profile_id"] = [f"TP3-{i+1:06d}" for i in range(len(profile_df))]

    def top_category_text(dist, n=3):
        if not dist:
            return ""
        return " ".join(f"{d['category']}({d['ratio']*100:.0f}%)" for d in dist[:n])

    profile_df["top_category_middle_text"] = profile_df["category_middle_dist"].map(top_category_text)

    # profile 1건 = 문서 1개 (임베딩 대상). 상품분류는 그룹 키가 아니라 "참고 정보"로만 들어갑니다.
    profile_df["trade_profile_search_text"] = (
        "[구매처분류] " + profile_df["buyer_type_path"] + "\n" +
        "[주문월] " + profile_df["order_month"].fillna("").astype(str) + "월\n" +
        "[주요 상품분류] " + profile_df["top_category_middle_text"] + "\n" +
        "[대표상품] " + profile_df["sample_products"].fillna("").astype(str) + "\n" +
        "[대표구매처] " + profile_df["sample_buyer_names"].fillna("").astype(str) + "\n" +
        "[행사/대상/시즌] " +
            profile_df["event_filter"].fillna("").astype(str) + " " +
            profile_df["target_filter"].fillna("").astype(str) + " " +
            profile_df["season_filter"].fillna("").astype(str) + "\n" +
        "[인쇄방법] " + profile_df["sample_print_methods"].fillna("").astype(str) + "\n" +
        "[거래수] " + profile_df["trade_count"].astype(str)
    ).map(clean_text)

    profile_df["trade_count_weight"] = np.log1p(profile_df["trade_count"])
    max_w = profile_df["trade_count_weight"].max()
    profile_df["trade_count_score"] = profile_df["trade_count_weight"] / max_w if max_w and max_w > 0 else 0.0

    days_since_last = (DATASET_MAX_DATE - profile_df["last_order_date"]).dt.days
    profile_df["days_since_last_order"] = days_since_last
    profile_df["recency_score"] = np.where(
        days_since_last.isna(), 0.3,  # 날짜 정보 없으면 중립값
        0.5 ** (days_since_last.fillna(0) / RECENCY_HALF_LIFE_DAYS)
    )

    return profile_df


trade_profile_df = make_trade_profiles_v3(trade_df)

print("원본 거래 row 수:", len(trade_df))
print("거래 profile 수:", len(trade_profile_df))
reduction = (1 - len(trade_profile_df) / max(len(trade_df), 1)) * 100
print(f"감소율: {reduction:.1f}%  (참고: 표본이 100건뿐이라 v2와 마찬가지로 감소율이 크지 않습니다)")
print("Profile당 평균 거래 수:", round(len(trade_df) / max(len(trade_profile_df), 1), 2))
print("1건 Profile 비율:", f"{(trade_profile_df['trade_count'] == 1).mean() * 100:.1f}%")
print("5건 이상 Profile 비율:", f"{(trade_profile_df['trade_count'] >= 5).mean() * 100:.1f}%")

display(trade_profile_df[[
    "trade_profile_id", "trade_count", "order_month", "buyer_type_path",
    "top_category_middle_text", "sample_products", "recency_score", "trade_count_score",
]].head(20))

## 7. 질문 조건 추출

*v1/v2와 동일한 함수입니다.*

In [ ]:
SEASON_MONTHS = {
    "봄": [3, 4, 5],
    "여름": [6, 7, 8],
    "가을": [9, 10, 11],
    "겨울": [12, 1, 2],
}


def extract_basic_conditions(query: str) -> dict:
    """정규식만으로 예산/수량/월/계절을 추출하는 fallback입니다."""
    q = str(query)

    budget = None
    m = re.search(r"(\d+(?:,\d+)*)\s*원", q)
    if m:
        budget = int(m.group(1).replace(",", ""))
    if budget is None:
        m = re.search(r"(\d+)\s*천\s*원", q)
        if m:
            budget = int(m.group(1)) * 1000
    if budget is None:
        m = re.search(r"(\d+)\s*만\s*원", q)
        if m:
            budget = int(m.group(1)) * 10000

    quantity = None
    m = re.search(r"(\d+(?:,\d+)*)\s*(?:개|ea|EA|pcs|PCS)", q)
    if m:
        quantity = int(m.group(1).replace(",", ""))

    event_month = None
    m = re.search(r"(1[0-2]|[1-9])\s*월", q)
    if m:
        event_month = int(m.group(1))

    season = next((s for s in SEASON_MONTHS if s in q), None)

    return {
        "raw_query": q,
        "buyer_context": "",
        "event_context": "",
        "purpose": "",
        "season": season,
        "event_month": event_month,
        "budget_max": budget,
        "quantity": quantity,
        "style_keywords": [],
        "must_have": [],
        "extraction_method": "regex_fallback",
    }


def safe_json_loads(text: str) -> dict:
    """LLM 응답에서 코드블록/잡음을 제거하고 JSON 객체만 파싱합니다."""
    text = str(text).strip()
    text = re.sub(r"^```json", "", text).strip()
    text = re.sub(r"^```", "", text).strip()
    text = re.sub(r"```$", "", text).strip()
    start, end = text.find("{"), text.rfind("}")
    if start >= 0 and end > start:
        text = text[start:end + 1]
    return json.loads(text)


def extract_query_condition(query: str, model: str = LLM_MODEL) -> dict:
    """Ollama LLM으로 질문을 구조화합니다. 실패하면 정규식 fallback을 씁니다."""
    fallback = extract_basic_conditions(query)

    try:
        import ollama
    except ModuleNotFoundError:
        return fallback

    system_prompt = """
너는 판촉물 상품 추천을 위한 조건 추출기다.
사용자 질문에서 추천에 필요한 조건만 JSON으로 추출한다.

주의:
- 상품 추천 사전을 만들지 않는다.
- "여름이면 선풍기"처럼 상품군을 임의 확장하지 않는다.
- 사용자가 말한 구매처, 행사, 목적, 계절, 월, 예산, 수량, 스타일만 구조화한다.
- 모르는 값은 null 또는 빈 문자열/빈 리스트로 둔다.
- 반드시 JSON만 출력한다.
"""

    user_prompt = f"""
아래 사용자 요청을 JSON으로 구조화해줘.

[사용자 요청]
{query}

[출력 JSON 스키마]
{{
  "buyer_context": "구매처/업종/기관/대상 예: 병원, 대학교, 기업, 어린이집",
  "event_context": "행사/상황 예: 개원, 창립기념, 박람회, 여름행사",
  "purpose": "용도 예: 답례품, 사은품, 홍보물, 기념품",
  "season": "봄/여름/가을/겨울 중 하나 또는 null",
  "event_month": 1~12 숫자 또는 null,
  "budget_max": 숫자 또는 null,
  "quantity": 숫자 또는 null,
  "style_keywords": ["실용적", "고급", "저렴한"] 형태,
  "must_have": ["로고인쇄", "휴대성"] 형태
}}
"""

    try:
        response = ollama.chat(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt.strip()},
                {"role": "user", "content": user_prompt.strip()},
            ],
            options={"temperature": 0.0, "num_ctx": 2048},
            stream=False,
        )
        parsed = safe_json_loads(response["message"]["content"])

        result = fallback.copy()
        for k, v in parsed.items():
            if k in result:
                result[k] = v

        for k in ["budget_max", "quantity", "event_month", "season"]:
            if result.get(k) in [None, "", []] and fallback.get(k) not in [None, "", []]:
                result[k] = fallback[k]

        result["raw_query"] = query
        result["extraction_method"] = "ollama_llm"
        return result

    except Exception as e:
        fallback["extraction_error"] = str(e)
        return fallback


# 테스트
for q in [
    "8월 행사에서 나눠줄 여름 판촉물 추천해줘",
    "병원 개원 답례품으로 3천원 이하 500개 추천해줘",
]:
    print("\nQUERY:", q)
    print(extract_query_condition(q))

## 8. 행사/사용월 기준 참고 주문월 계산

*v1/v2와 동일한 로직입니다.*

In [ ]:
def prev_month(month: int, n: int = 1) -> int:
    m = month - n
    while m <= 0:
        m += 12
    return m


def get_month_weights(event_month: int) -> dict:
    if event_month is None or pd.isna(event_month):
        return {}
    event_month = int(event_month)
    return {
        prev_month(event_month, 1): 1.00,
        prev_month(event_month, 2): 0.85,
        event_month: 0.45,
        prev_month(event_month, 3): 0.25,
    }


def get_event_months(query_condition: dict) -> list:
    months = []
    event_month = query_condition.get("event_month")
    if event_month not in [None, "", np.nan]:
        try:
            months.append(int(event_month))
        except Exception:
            pass

    season = query_condition.get("season")
    if season in SEASON_MONTHS:
        months.extend(SEASON_MONTHS[season])

    return sorted(set(months))


def get_reference_month_weights(query_condition: dict) -> dict:
    combined = {}
    for m in get_event_months(query_condition):
        for month, score in get_month_weights(m).items():
            combined[month] = max(combined.get(month, 0), score)
    return combined


def calc_date_lead_score(order_month, query_condition: dict) -> float:
    if pd.isna(order_month):
        return 0.0
    weights = get_reference_month_weights(query_condition)
    if not weights:
        return 0.0
    try:
        return float(weights.get(int(order_month), 0.0))
    except Exception:
        return 0.0


# 테스트
for q in ["8월 행사 판촉물", "여름 행사 판촉물", "12월 연말 선물"]:
    cond = extract_basic_conditions(q)
    print(q, "->", get_reference_month_weights(cond))

## 9. 임베딩 생성 (상품 row + 거래 Profile)

*상품은 v1/v2와 동일하게 row 단위입니다. 거래 Profile 텍스트만 6번 섹션에서 새로 만든
`trade_profile_search_text`로 바뀝니다.*

In [ ]:
def try_ollama_embed_texts(texts, model=EMBED_MODEL, batch_size=32):
    """Ollama Python 패키지의 embed()/embeddings() API 차이를 모두 지원합니다."""
    import ollama

    all_embeddings = []
    for start in range(0, len(texts), batch_size):
        batch = [str(x) for x in texts[start:start + batch_size]]
        try:
            resp = ollama.embed(model=model, input=batch)
            if "embeddings" in resp:
                all_embeddings.extend(resp["embeddings"])
                continue
        except Exception:
            pass

        for text in batch:
            try:
                resp = ollama.embeddings(model=model, prompt=text)
                all_embeddings.append(resp["embedding"])
            except Exception as e:
                raise RuntimeError(f"Ollama embedding 실패: {e}")

    return np.array(all_embeddings, dtype=np.float32)


def build_search_index(name, texts, use_ollama=True, model=EMBED_MODEL):
    """임베딩 인덱스를 만듭니다. 캐시가 있으면 재사용합니다."""
    texts = [str(x) for x in texts]
    cache_path = CACHE_DIR / f"{name}_{model}_{text_hash(texts)}.npy"

    if use_ollama:
        try:
            if cache_path.exists():
                matrix = normalize_vector_matrix(np.load(cache_path))
                print(f"[{name}] 임베딩 캐시 로딩: {cache_path.name}")
                return {"backend": "ollama", "model": model, "matrix": matrix, "vectorizer": None}

            print(f"[{name}] Ollama 임베딩 생성 중... rows={len(texts)}")
            matrix = try_ollama_embed_texts(texts, model=model)
            np.save(cache_path, matrix)
            return {"backend": "ollama", "model": model, "matrix": normalize_vector_matrix(matrix), "vectorizer": None}

        except Exception as e:
            warnings.warn(f"[{name}] Ollama 임베딩 실패, TF-IDF로 대체합니다: {e}")

    print(f"[{name}] TF-IDF fallback 생성")
    vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=1)
    matrix = vectorizer.fit_transform(texts)
    return {"backend": "tfidf", "model": "tfidf_char_ngram", "matrix": matrix, "vectorizer": vectorizer}


def search_index(query_text, index) -> np.ndarray:
    """인덱스 backend에 맞춰 query와 문서들 간 유사도 점수를 계산합니다."""
    if index["backend"] == "ollama":
        q_vec = normalize_vector_matrix(try_ollama_embed_texts([query_text], model=index["model"]))
        return cosine_scores(q_vec[0], index["matrix"])
    if index["backend"] == "tfidf":
        q_vec = index["vectorizer"].transform([query_text])
        return cosine_similarity(q_vec, index["matrix"]).flatten()
    raise ValueError(f"지원하지 않는 backend: {index['backend']}")


# 상품: row 단위 (v1/v2와 동일 — 캐시도 그대로 재사용됩니다)
product_index = build_search_index("product", product_df["product_search_text"], use_ollama=USE_OLLAMA_EMBEDDING)

# 거래: profile 단위 (v3에서 텍스트 구성이 바뀌어 캐시가 새로 생성됩니다)
trade_profile_index = build_search_index("trade_profile_v3", trade_profile_df["trade_profile_search_text"], use_ollama=USE_OLLAMA_EMBEDDING)

print("product backend:", product_index["backend"])
print("trade_profile backend:", trade_profile_index["backend"])

## 10. 유사 거래 Profile 검색

*RAG 단계: Retrieval — v2의 `retrieve_similar_trade_profiles`에 대응합니다.
recency_score가 추가되고, 거래량 점수(trade_count) 비중은 낮춰서 "인기라서 상위"가
되는 정도를 줄였습니다.*

| 점수 | v2 비중 | v3 비중 |
|---|---:|---:|
| 의미유사도 | 50% | 45% |
| 날짜 리드타임 | 20% | 15% |
| 예산/수량/시즌 조건 | 15% | 15% |
| 거래량 점수 | 15% | 10% |
| **최근 거래 가중치** | 없음 | **15%** |

In [ ]:
def build_query_text(query_condition: dict) -> str:
    """query_condition(구조화된 질문)을 검색 쿼리 문자열로 합칩니다."""
    parts = [
        query_condition.get("raw_query", ""),
        query_condition.get("buyer_context", ""),
        query_condition.get("event_context", ""),
        query_condition.get("purpose", ""),
        query_condition.get("season", ""),
        " ".join(query_condition.get("style_keywords") or []),
        " ".join(query_condition.get("must_have") or []),
    ]
    return combine_text(*parts)


def calc_profile_condition_score(row, query_condition: dict) -> float:
    """예산/수량/시즌이 이 거래 profile과 얼마나 맞는지 0~1 점수로 계산합니다."""
    score = 0.0

    budget = query_condition.get("budget_max")
    if budget not in [None, ""] and not pd.isna(budget):
        prices = [p for p in [row.get("median_bulk_price"), row.get("median_middle_price"), row.get("median_small_price")] if not pd.isna(p)]
        if prices:
            min_price = min(prices)
            if min_price <= float(budget):
                score += 0.35
            elif min_price <= float(budget) * 1.2:
                score += 0.15

    quantity = query_condition.get("quantity")
    if quantity not in [None, ""] and not pd.isna(quantity):
        moq = row.get("median_moq", np.nan)
        if pd.isna(moq) or moq <= float(quantity):
            score += 0.25

    season = query_condition.get("season")
    if season and season in str(row.get("season_filter", "")):
        score += 0.20

    return min(score, 1.0)


# v3: 거래신호(trade_count)와 recency 비중을 낮게 잡아, Profile 총점이 상품 재정렬 단계에서
# 너무 강한 신호로 쓰이지 않도록 합니다 (역할 분리의 첫 단계).
PROFILE_SCORE_WEIGHTS = {
    "semantic": 0.45,
    "date_lead": 0.15,
    "condition": 0.15,
    "trade_count": 0.10,
    "recency": 0.15,
}


def retrieve_similar_trade_profiles_v3(query: str, query_condition: dict, top_k: int = 20) -> pd.DataFrame:
    """질문과 가장 관련 있는 거래 profile top_k건을 점수와 함께 돌려줍니다."""
    query_text = build_query_text(query_condition)
    semantic_scores = search_index(query_text, trade_profile_index)

    result = trade_profile_df.copy()
    result["profile_semantic_score"] = semantic_scores
    result["date_lead_score"] = result["order_month"].apply(lambda m: calc_date_lead_score(m, query_condition))
    result["profile_condition_score"] = result.apply(lambda row: calc_profile_condition_score(row, query_condition), axis=1)

    w = PROFILE_SCORE_WEIGHTS
    result["profile_total_score"] = (
        result["profile_semantic_score"] * w["semantic"] +
        result["date_lead_score"] * w["date_lead"] +
        result["profile_condition_score"] * w["condition"] +
        result["trade_count_score"] * w["trade_count"] +
        result["recency_score"] * w["recency"]
    )

    return result.sort_values("profile_total_score", ascending=False).head(top_k).reset_index(drop=True)


# 테스트
test_query = "8월 행사에서 나눠줄 여름 판촉물 추천해줘"
test_condition = extract_query_condition(test_query)
similar_profiles = retrieve_similar_trade_profiles_v3(test_query, test_condition, top_k=10)

print("[추출된 조건]")
print(json.dumps(test_condition, ensure_ascii=False, indent=2))

display(similar_profiles[[
    "trade_profile_id", "profile_total_score", "profile_semantic_score",
    "date_lead_score", "profile_condition_score", "trade_count_score", "recency_score",
    "buyer_type_path", "top_category_middle_text", "sample_products",
]])

## 11. Profile별 상품분류 비율 신호 (v3 핵심 2)

*RAG 단계: Retrieval 결과 재구성 — v2는 "신호 텍스트"를 만들어 상품과 다시 임베딩 비교했는데,
이 방식은 과거 인기상품이 그대로 재추천되는 경향을 만들었습니다(README 6.3, "거래량이 여러
단계에서 반복 반영"). v3는 특정 상품명이 아니라 **"상품군 선호도"만 신호로 씁니다** —
유사 Profile들의 상품분류 비율을 profile_total_score로 가중합해서 카테고리별 선호 점수를
만들고, 이 값만 다음 단계(12번)에서 순위 보정에 씁니다.*

노트북 인트로에서 설명했듯 거래데이터와 상품데이터의 상품분류 체계가 서로 달라서,
카테고리 문자열을 정확히 일치시키지 않고 상품 텍스트 안에 포함되는지로 겹침을 판단합니다.

In [ ]:
def aggregate_category_preference(similar_profiles: pd.DataFrame, dist_col: str) -> dict:
    """유사 profile들의 상품분류 비율을 profile_total_score로 가중합해 카테고리별 선호도로 만듭니다."""
    pref = {}
    for _, row in similar_profiles.iterrows():
        weight = float(row.get("profile_total_score", 0))
        for item in (row.get(dist_col) or []):
            cat = item["category"]
            pref[cat] = pref.get(cat, 0.0) + weight * item["ratio"]

    if not pref:
        return {}
    max_v = max(pref.values())
    if max_v <= 0:
        return {}
    return {cat: v / max_v for cat, v in pref.items()}


def calc_profile_boost_score(row, category_pref_middle: dict, category_pref_small: dict) -> float:
    """거래/상품 분류체계가 달라 정확 일치가 거의 없으므로, 선호 카테고리 문자열이
    상품 텍스트(카테고리경로+상품명+키워드) 안에 등장하는지로 겹침 점수를 계산합니다."""
    product_text = combine_text(row.get("category_path", ""), row.get("product_name", ""), row.get("keywords", ""))

    def matched_score(pref: dict, weight: float) -> float:
        if not pref or not product_text:
            return 0.0
        best = 0.0
        for cat, score in pref.items():
            if cat and cat in product_text:
                best = max(best, score)
        return best * weight

    return min(matched_score(category_pref_middle, 0.65) + matched_score(category_pref_small, 0.35), 1.0)


# 테스트
category_pref_middle = aggregate_category_preference(similar_profiles, "category_middle_dist")
category_pref_small = aggregate_category_preference(similar_profiles, "category_small_dist")

print("[상위 카테고리 선호 (중분류)]")
print(dict(sorted(category_pref_middle.items(), key=lambda x: -x[1])[:8]))

## 12. 현재 상품 하드 필터 (v3 핵심 3)

*RAG 단계: Retrieval 이전 필터링 — v2는 예산/MOQ를 위반해도 soft score로 감점만 줘서,
다른 점수가 높으면 조건 위반 상품이 여전히 상위에 남을 수 있었습니다(README 5.5).
v3는 판매상태·가격·MOQ를 검색 이전 단계에서 하드 필터로 적용합니다.
단, 필터가 너무 강해서 후보가 0건이 되는 걸 막기 위해 단계적으로 완화합니다
(그 완화 과정을 `filter_log`로 남깁니다 — 왜 후보가 이렇게 좁혀졌는지 추적 가능하게).*

In [ ]:
def filter_products_hard(product_df: pd.DataFrame, budget, quantity) -> tuple:
    """(필터링된 product_df, 적용된 완화 단계 로그)를 반환합니다."""
    log = []
    df = product_df[~product_df["is_sold_out"]].copy()
    log.append(f"판매상태 필터: {len(product_df)} -> {len(df)}건 (품절 확정 상품만 제외)")

    if budget not in [None, ""] and not pd.isna(budget):
        budget = float(budget)
        for ratio, label in [(1.0, "예산 이내"), (1.2, "예산 x1.2까지 완화"), (1.5, "예산 x1.5까지 완화")]:
            candidate = df[df["price"].isna() | (df["price"] <= budget * ratio)]
            if len(candidate) > 0:
                df = candidate
                log.append(f"가격 필터({label}): {len(df)}건 남음")
                break
        else:
            log.append("가격 필터: 조건을 만족하는 상품이 없어 필터를 적용하지 않음")

    if quantity not in [None, ""] and not pd.isna(quantity):
        quantity = float(quantity)
        for ratio, label in [(1.0, "MOQ 이내"), (1.5, "MOQ x1.5까지 완화")]:
            candidate = df[df["moq"].isna() | (df["moq"] <= quantity * ratio)]
            if len(candidate) > 0:
                df = candidate
                log.append(f"MOQ 필터({label}): {len(df)}건 남음")
                break
        else:
            log.append("MOQ 필터: 조건을 만족하는 상품이 없어 필터를 적용하지 않음")

    if len(df) == 0:
        df = product_df.copy()
        log.append("경고: 모든 필터 완화 후에도 후보가 0건이라 전체 상품으로 되돌림")

    return df, log


# 테스트
filtered_products, filter_log = filter_products_hard(product_df, budget=3000, quantity=500)
for line in filter_log:
    print("-", line)
print("최종 후보 수:", len(filtered_products))

## 13. 관련도 임계값 + 다양성 조정 (v3 핵심 4)

*v2는 top_k로 결과를 자르기만 해서, 관련도가 낮은 상품도 top_k 안에 들어가면 그대로
추천됐습니다(README 6.3 "관련도 임계값과 정량 평가체계 부족"). v3는 최고점 대비 상대
비율로 임계값을 적용합니다 — Ollama 임베딩과 TF-IDF는 점수 스케일이 완전히 달라서
절대값 기준(예: 0.3 이상)을 쓰면 backend에 따라 전혀 다르게 동작하기 때문입니다.
다양성 조정은 같은 소분류 상품이 top_k를 도배하지 않도록 카테고리당 개수를 제한합니다
(README 13 "중복 제거 및 다양성 조정" 과제).*

In [ ]:
RELEVANCE_RATIO_STEPS = [0.5, 0.3, 0.15, 0.0]  # 최고점 대비 비율. 0.0은 "임계값 사실상 해제"
MAX_PER_CATEGORY_SMALL = 2


def apply_relevance_threshold(scored_df: pd.DataFrame, score_col: str, top_k: int) -> pd.DataFrame:
    """최고점 대비 비율로 관련도 낮은 후보를 제외합니다. top_k를 못 채우면 단계적으로 완화합니다."""
    if len(scored_df) == 0:
        return scored_df
    max_score = scored_df[score_col].max()
    if max_score <= 0:
        return scored_df
    for ratio in RELEVANCE_RATIO_STEPS:
        candidate = scored_df[scored_df[score_col] >= max_score * ratio]
        if len(candidate) >= top_k or ratio == RELEVANCE_RATIO_STEPS[-1]:
            return candidate
    return scored_df


def select_with_diversity(sorted_df: pd.DataFrame, top_k: int, max_per_category=MAX_PER_CATEGORY_SMALL) -> pd.DataFrame:
    """같은 소분류가 top_k를 도배하지 않도록 카테고리당 최대 개수를 제한합니다."""
    picked_idx = []
    category_counts = {}
    for idx, row in sorted_df.iterrows():
        cat = row.get("category_small", "")
        if category_counts.get(cat, 0) >= max_per_category:
            continue
        picked_idx.append(idx)
        category_counts[cat] = category_counts.get(cat, 0) + 1
        if len(picked_idx) >= top_k:
            break
    if len(picked_idx) < top_k:
        for idx in sorted_df.index:
            if idx not in picked_idx:
                picked_idx.append(idx)
            if len(picked_idx) >= top_k:
                break
    return sorted_df.loc[picked_idx]

## 14. 상품 후보 랭킹 (역할 분리: 질문 검색이 중심, Profile은 보정)

*RAG 단계: Ranking — v2는 거래신호 비중(32%+22%=54%)이 질문 자체(18%)보다 커서, 과거
인기 패턴이 현재 질문보다 결과를 더 강하게 이끌었습니다(README 6.3). v3는 이 비중을
뒤집어 질문 자체 유사도를 가장 크게 둡니다.*

| 점수 | v2 비중 | v3 비중 |
|---|---:|---:|
| query_product_score (질문-상품 유사도) | 18% | **50%** |
| profile_boost_score (Profile 카테고리 선호, 12번 산출) | 32%+22%=54%\* | **20%** |
| budget_score | 15% | 12% |
| quantity_score | 5% | 6% |
| product_quality_score | 8% | 12% |

\* v2는 `trade_signal_product_score`(32%)와 `profile_category_signal_score`(22%) 두 개로
나뉘어 있었는데, v3는 이를 `profile_boost_score` 하나로 합치고 비중을 크게 낮췄습니다.

In [ ]:
RANK_WEIGHTS = {
    "query": 0.50,
    "profile_boost": 0.20,
    "budget": 0.12,
    "quantity": 0.06,
    "quality": 0.12,
}


def calc_budget_score(price, budget):
    if budget in [None, ""] or pd.isna(budget):
        return 0.5
    if pd.isna(price):
        return 0.2
    price, budget = float(price), float(budget)
    if price <= budget:
        return 1.0
    if price <= budget * 1.2:
        return 0.65
    if price <= budget * 1.5:
        return 0.35
    return 0.0


def calc_quantity_score(moq, quantity):
    if quantity in [None, ""] or pd.isna(quantity):
        return 0.5
    if pd.isna(moq):
        return 0.7
    moq, quantity = float(moq), float(quantity)
    if moq <= quantity:
        return 1.0
    if moq <= quantity * 1.5:
        return 0.5
    return 0.0


def calc_product_quality_score(row):
    score = 0.0
    if clean_text(row.get("product_name", "")):
        score += 0.25
    if clean_text(row.get("category_path", "")):
        score += 0.25
    if not pd.isna(row.get("price", np.nan)):
        score += 0.25
    if clean_text(row.get("keywords", "")) or clean_text(row.get("summary", "")):
        score += 0.25
    return score


def make_recommend_reason(row, budget, quantity) -> str:
    reasons = []
    if row["query_product_score"] >= 0.55:
        reasons.append("질문과 상품 설명의 의미 유사도 높음")
    elif row["query_product_score"] >= 0.3:
        reasons.append("질문과 상품 설명이 일부 관련 있음")

    if row["profile_boost_score"] >= 0.5:
        reasons.append("유사 구매상황에서 선호된 상품군과 일치")
    elif row["profile_boost_score"] > 0:
        reasons.append("유사 구매상황 선호 상품군과 일부 겹침")

    if budget not in [None, ""] and not pd.isna(budget) and pd.notna(row["price"]):
        reasons.append(f"예산 {int(float(budget)):,}원 이하 조건 충족" if float(row["price"]) <= float(budget) else "예산 초과 여부 확인 필요")

    if quantity not in [None, ""] and not pd.isna(quantity):
        moq_ok = pd.isna(row["moq"]) or float(row["moq"]) <= float(quantity)
        reasons.append("요청 수량 기준 최소구매수량 충족 가능" if moq_ok else "최소구매수량 확인 필요")

    return " / ".join(reasons) if reasons else "현재 상품데이터 기준 추천 후보로 검토 가능"


def rank_products_v3(query_condition: dict, candidate_df: pd.DataFrame, category_pref_middle: dict, category_pref_small: dict, top_k: int = 5) -> pd.DataFrame:
    query_text = build_query_text(query_condition)

    # product_index는 전체 상품 기준으로 만들어져 있으므로, 하드 필터로 좁혀진
    # candidate_df의 원래 위치(index)로 점수를 그대로 골라 씁니다 (재임베딩 불필요).
    all_query_scores = search_index(query_text, product_index)
    result = candidate_df.copy()
    result["query_product_score"] = all_query_scores[result.index.to_numpy()]

    result["profile_boost_score"] = result.apply(lambda row: calc_profile_boost_score(row, category_pref_middle, category_pref_small), axis=1)

    budget = query_condition.get("budget_max")
    quantity = query_condition.get("quantity")
    result["budget_score"] = result["price"].apply(lambda p: calc_budget_score(p, budget))
    result["quantity_score"] = result["moq"].apply(lambda q: calc_quantity_score(q, quantity))
    result["product_quality_score"] = result.apply(calc_product_quality_score, axis=1)

    w = RANK_WEIGHTS
    result["final_score"] = (
        result["query_product_score"] * w["query"] +
        result["profile_boost_score"] * w["profile_boost"] +
        result["budget_score"] * w["budget"] +
        result["quantity_score"] * w["quantity"] +
        result["product_quality_score"] * w["quality"]
    )

    result = apply_relevance_threshold(result, "query_product_score", top_k)
    result = result.sort_values("final_score", ascending=False)
    result = select_with_diversity(result, top_k)

    result["recommend_reason"] = result.apply(lambda row: make_recommend_reason(row, budget, quantity), axis=1)

    output_cols = [
        "product_id", "product_name", "price", "moq", "category_path", "keywords",
        "query_product_score", "profile_boost_score", "budget_score", "quantity_score",
        "product_quality_score", "final_score", "recommend_reason", "image",
    ]
    return result[output_cols].reset_index(drop=True)

## 15. 최종 추천 함수

*RAG 단계 전체 통합 — v2의 `recommend_products_v2()`에 대응합니다.*

```text
질문 → 조건 추출 → 현재 상품 하드 필터 → 유사 거래 Profile 검색
     → Profile 카테고리 비율 신호 → 상품 랭킹(질문 중심 + 관련도 임계값 + 다양성)
```

In [ ]:
def recommend_products_v3(query: str, top_k: int = DEFAULT_TOP_K, profile_top_k: int = 30) -> dict:
    query_condition = extract_query_condition(query)
    budget = query_condition.get("budget_max")
    quantity = query_condition.get("quantity")

    filtered_products, filter_log = filter_products_hard(product_df, budget, quantity)

    similar_profiles = retrieve_similar_trade_profiles_v3(query, query_condition, top_k=profile_top_k)
    category_pref_middle = aggregate_category_preference(similar_profiles, "category_middle_dist")
    category_pref_small = aggregate_category_preference(similar_profiles, "category_small_dist")

    recommendations = rank_products_v3(query_condition, filtered_products, category_pref_middle, category_pref_small, top_k=top_k)

    return {
        "query": query,
        "query_condition": query_condition,
        "reference_month_weights": get_reference_month_weights(query_condition),
        "filter_log": filter_log,
        "filtered_product_count": len(filtered_products),
        "similar_profiles": similar_profiles,
        "category_pref_middle": category_pref_middle,
        "category_pref_small": category_pref_small,
        "recommendations": recommendations,
        "backend": {
            "product": product_index["backend"],
            "trade_profile": trade_profile_index["backend"],
        },
        "counts": {
            "trade_rows": len(trade_df),
            "trade_profiles": len(trade_profile_df),
            "products": len(product_df),
        },
    }


# 테스트
test_query = "8월 행사에서 나눠줄 여름 판촉물 추천해줘"
result = recommend_products_v3(test_query, top_k=5, profile_top_k=20)

print("[질문]", result["query"])
print("\n[추출된 조건]")
print(json.dumps(result["query_condition"], ensure_ascii=False, indent=2))
print("\n[필터 로그]")
for line in result["filter_log"]:
    print(" -", line)
print("\n[사용 backend]", result["backend"])
print("[데이터 수]", result["counts"])

print("\n[추천 상품]")
display(result["recommendations"])

## 16. 여러 질문 일괄 테스트 및 결과 저장

In [ ]:
test_queries = [
    "8월 행사에서 나눠줄 여름 판촉물 추천해줘",
    "병원 개원 답례품으로 3천원 이하 500개 추천해줘",
    "대학교 OT에서 나눠줄 저렴한 사은품 추천해줘",
    "회사 창립기념품으로 실용적인 상품 추천해줘",
    "박람회 부스에서 나눠줄 홍보물 추천해줘",
    "12월 연말 고객 선물로 5만원 이하 상품 추천해줘",
    "어린이집 행사 답례품 추천해줘",
    "로고 인쇄 가능한 텀블러 추천해줘",
]

all_rows = []
for query in test_queries:
    r = recommend_products_v3(query, top_k=5, profile_top_k=20)
    cond = r["query_condition"]

    for rank, (_, row) in enumerate(r["recommendations"].iterrows(), start=1):
        all_rows.append({
            "query": query,
            "rank": rank,
            "buyer_context": cond.get("buyer_context", ""),
            "event_context": cond.get("event_context", ""),
            "purpose": cond.get("purpose", ""),
            "season": cond.get("season", ""),
            "event_month": cond.get("event_month", ""),
            "product_id": row["product_id"],
            "product_name": row["product_name"],
            "price": row["price"],
            "moq": row["moq"],
            "category_path": row["category_path"],
            "query_product_score": row["query_product_score"],
            "profile_boost_score": row["profile_boost_score"],
            "final_score": row["final_score"],
            "recommend_reason": row["recommend_reason"],
            "filtered_product_count": r["filtered_product_count"],
            "trade_rows": r["counts"]["trade_rows"],
            "trade_profiles": r["counts"]["trade_profiles"],
        })

batch_result_df = pd.DataFrame(all_rows)
batch_result_path = OUTPUT_DIR / "product_recommendation_v3_260724_result.xlsx"
batch_result_df.to_excel(batch_result_path, index=False)

print("결과 저장:", batch_result_path)
display(batch_result_df.head(20))

## 17. Ollama로 고객용 추천 답변 생성 + 후보 ID 검증 (v3 핵심 5)

*v1/v2와 같은 구조로 답변을 생성하되, v3는 **생성된 답변에 추천 후보 외 상품이 슬쩍
섞였는지**를 코드로 검증합니다(README "LLM이 부정확한 후보를 자연스럽게 설명할 수 있음"
문제에 대한 보완). `verify_recommendation_ids`는 실제 LLM 호출 없이도 텍스트만 있으면
검증할 수 있어서, 아래에 Ollama 없이도 확인 가능한 예시를 먼저 둡니다.*

In [ ]:
def build_answer_context(result: dict) -> str:
    """LLM에게 넘길 컨텍스트를 하나의 텍스트로 구성합니다."""
    lines = ["[질문]", result["query"], ""]
    lines += ["[추출된 조건]", json.dumps(result["query_condition"], ensure_ascii=False), ""]
    lines += ["[참고 주문월 가중치]", json.dumps(result["reference_month_weights"], ensure_ascii=False), ""]
    lines += ["[적용된 필터]"] + [f"- {line}" for line in result["filter_log"]] + [""]

    lines.append("[추천 상품 후보]")
    for i, (_, row) in enumerate(result["recommendations"].iterrows(), start=1):
        lines += [
            f"{i}. {row['product_name']}",
            f"   - 가격: {row['price']}",
            f"   - 최소구매수량: {row['moq']}",
            f"   - 카테고리: {row['category_path']}",
            f"   - 추천근거: {row['recommend_reason']}",
            "",
        ]

    return "\n".join(lines)


def generate_customer_answer(query: str, model: str = LLM_MODEL, top_k: int = 5) -> dict:
    import ollama

    result = recommend_products_v3(query, top_k=top_k, profile_top_k=30)
    context = build_answer_context(result)

    system_prompt = """
너는 쇼핑몰 판촉물 상품 추천 어시스턴트다.

규칙:
- 제공된 추천 후보만 근거로 답변한다. 후보에 없는 상품명을 만들어내지 않는다.
- 데이터에 없는 납기, 재고, 인쇄 가능 여부는 단정하지 않는다.
- 가격과 최소구매수량은 제공된 값만 말한다.
- 고객에게 보여줄 수 있게 자연스러운 한국어로 답변한다.
- 상위 3~5개 상품을 추천하고, 각 상품의 추천 이유를 짧게 적는다.
- 마지막에 확인이 필요한 조건을 적는다.
"""

    response = ollama.chat(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt.strip()},
            {"role": "user", "content": f"아래 근거를 바탕으로 고객에게 상품 추천 답변을 작성해줘.\n\n{context}"},
        ],
        options={"temperature": 0.2, "num_ctx": 4096},
        stream=False,
    )

    return {"answer": response["message"]["content"], "result": result}


def verify_recommendation_ids(answer_text: str, recommendations: pd.DataFrame, full_product_df: pd.DataFrame) -> dict:
    """LLM 답변에 추천 후보 목록 밖의 상품명이 등장하는지 확인합니다 (환각 방지)."""
    recommended_names = set(recommendations["product_name"].tolist())
    all_names = full_product_df["product_name"].tolist()

    mentioned_in_answer = {name for name in all_names if name and name in answer_text}
    unexpected = mentioned_in_answer - recommended_names
    not_mentioned = recommended_names - mentioned_in_answer

    return {
        "mentioned_count": len(mentioned_in_answer),
        "unexpected_products": sorted(unexpected),
        "recommended_but_not_mentioned": sorted(not_mentioned),
        "is_clean": len(unexpected) == 0,
    }


# Ollama 없이도 확인 가능한 검증 데모 (실제 LLM 답변이 아니라 예시 텍스트)
demo_result = recommend_products_v3("8월 행사에서 나눠줄 여름 판촉물 추천해줘", top_k=5)
ok_names = demo_result["recommendations"]["product_name"].tolist()[:2]
fake_answer_ok = f"추천 상품은 다음과 같습니다: {ok_names[0]}, {ok_names[1]} 등입니다."
fake_answer_bad = fake_answer_ok + " 그리고 " + product_df["product_name"].iloc[0] + "도 좋습니다."

print("[정상 케이스] 후보 안에서만 언급:")
print(verify_recommendation_ids(fake_answer_ok, demo_result["recommendations"], product_df))
print("\n[문제 케이스] 후보 밖 상품이 섞임:")
print(verify_recommendation_ids(fake_answer_bad, demo_result["recommendations"], product_df))

# 실제 LLM 답변 생성 + 검증 (Ollama가 실행 중이어야 동작합니다)
try:
    answer_result = generate_customer_answer("8월 행사에서 나눠줄 여름 판촉물 추천해줘", top_k=5)
    print("\n[LLM 답변]")
    print(answer_result["answer"])
    verify_report = verify_recommendation_ids(answer_result["answer"], answer_result["result"]["recommendations"], product_df)
    print("\n[ID 검증 결과]")
    print(verify_report)
except Exception as e:
    print("\nOllama 미실행 등으로 실제 답변 생성은 건너뜁니다:", e)

## 18. 평가셋과 Hit@5 (v3 핵심 6)

*README 12장 평가 질문 4개를 기준으로, 카테고리 키워드가 상품 `category_path`/`keywords`/
`product_name`에 포함되는지로 정답(ground truth)을 **자동 생성**했습니다. Hit@k 계산 자체는
자동/객관적이지만, 이 정답 목록이 타당한지는 사람이 한 번 검토해야 합니다 —
지금은 카테고리 키워드 기반이라 느슨합니다. 나중에 상품 ID 단위로 더 엄격하게
큐레이션하면 지표의 신뢰도가 올라갑니다.*

In [ ]:
EVAL_QUERIES = [
    {"query": "병원 개원 답례품으로 3천 원 이하 500개 추천해줘", "expected_keywords": ["타월", "위생", "칫솔"]},
    {"query": "대학생 OT에서 나눠줄 실용적인 기념품이 필요해", "expected_keywords": ["보조배터리", "텀블러", "문구", "볼펜", "다이어리"]},
    {"query": "8월 야외 행사에 어울리는 여름 판촉물을 추천해줘", "expected_keywords": ["선풍기", "우산", "부채"]},
    {"query": "VIP 고객에게 줄 고급스러운 선물을 찾고 있어", "expected_keywords": ["텀블러", "상패", "액세서리", "감사패"]},
]


def build_ground_truth_by_keywords(product_df: pd.DataFrame, keywords: list) -> list:
    """카테고리 키워드가 상품 텍스트에 포함되면 정답 후보로 간주합니다 (1차 자동 초안)."""
    text = (product_df["category_path"] + " " + product_df["product_name"] + " " + product_df["keywords"])
    mask = text.apply(lambda t: any(kw in t for kw in keywords))
    return product_df.loc[mask, "product_id"].tolist()


def evaluate_hit_at_k(eval_queries: list, k: int = 5) -> pd.DataFrame:
    rows = []
    for item in eval_queries:
        gt_ids = set(build_ground_truth_by_keywords(product_df, item["expected_keywords"]))
        r = recommend_products_v3(item["query"], top_k=k, profile_top_k=20)
        top_ids = set(r["recommendations"]["product_id"].tolist())
        hit = len(gt_ids & top_ids) > 0
        rows.append({
            "query": item["query"],
            "expected_keywords": ", ".join(item["expected_keywords"]),
            "ground_truth_count": len(gt_ids),
            f"hit@{k}": hit,
        })
    return pd.DataFrame(rows)


eval_result = evaluate_hit_at_k(EVAL_QUERIES, k=5)
display(eval_result)
print(f"Hit@5 (전체 평균): {eval_result['hit@5'].mean() * 100:.1f}%")

# 다음 단계

이 v3(현재 상품 중심 + Profile 순위 보정)를 실행한 뒤 확인할 것:

1. `필터 로그`를 보고, 예산/MOQ 조건이 실제로 몇 단계까지 완화됐는지 — 자주 완화된다면
   가격/MOQ 데이터 자체나 완화 비율(1.2, 1.5)을 다시 검토할 필요가 있습니다.
2. `profile_boost_score`가 0에 가까운 상품이 많다면, 거래-상품 분류체계 매핑을 더
   정교하게 만들 여지가 있는지 확인합니다.
3. `query_product_score` 비중(50%)이 실제로 추천 결과를 주도하는지, v2 결과와 같은
   질문으로 비교했을 때 "과거 인기상품 쏠림"이 줄었는지 확인합니다.
4. Hit@5 평가셋의 카테고리 키워드가 실제로 타당한지 검토하고, 가능하면 상품 ID 단위
   정답셋으로 업그레이드합니다.

## 다음 버전

다음은 v4입니다 — 하이브리드 검색(BM25+벡터)과 리랭커를 도입해서 "검색 자체의 품질"을
높이는 단계입니다. v3에서 갖춘 평가셋(Hit@5)으로 v4의 개선 여부를 직접 비교할 수 있습니다.
전체 커리큘럼은 `docs/product_recommendation_RAG_커리큘럼_260724.md`를 참고하세요.